In [ ]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from torch_geometric.loader import DataLoader

from scripts.data_formatting import SmilesDataset
from scripts.downstream import get_prediction_smiles, split_batch_by_molecule
from scripts.nn_models import GINE
from scripts.test import mapped_molecules_equivalent

In [2]:
# raw dataset
df = pd.read_csv('../datasets/13k_All_Manual.csv')
smiles_list = df['SMILES Labelled'].tolist()

In [3]:
# 70/15/15 train/test/val split
train_smiles, test_smiles = train_test_split(smiles_list, test_size=0.15, random_state=42)
train_smiles, val_smiles = train_test_split(train_smiles, test_size=0.1765, random_state=42)

# dataset objects
train_dataset = SmilesDataset(train_smiles)
test_dataset  = SmilesDataset(test_smiles)
val_dataset   = SmilesDataset(val_smiles)

# dataloaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)

4
4
4
4
4
4
2
4
2
4
2
3
4
4
4
4
4
3
4
4
4
4
4
4
4
2
4
4
3
4
2
2
4
4
4
4
4
4
4
4
4
3
4
4
4
3
4
4
4
4
4
4
4
4
4
4
2
4
2
3
4
4
2
4
4
4
3
4
2
3
4
4
4
4
4
2
4
4
4
2
4
2
4
4
1
2
6
2
2
4
4
4
6
2
2
4
4
4
4
4
4
4
4
2
1
1
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
1
3
2
4
4
4
1
1
1
1
4
4
2
4
2
4
4
4
4
3
4
4
2
4
4
2
2
4
4
4
4
2
3
4
4
4
4
4
4
4
4
4
4
1
2
4
4
2
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
2
6
2
2
4
1
4
4
4
4
4
4
4
4
4
4
4
2
4
4
4
4
4
2
4
4
1
4
4
4
4
4
2
4
2
2
4
4
4
4
1
3
4
3
3
3
4
4
4
3
4
2
3
4
2
4
4
4
4
2
4
2
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
2
3
3
4
4
4
4
4
4
2
4
4
4
3
4
4
4
3
4
2
3
4
2
4
4
4
4
2
4
2
4
4
4
4
4
4
4
4
4
4
2
1
5
4
1
1
1
4
4
4
4
4
4
4
4
4
4
6
2
2
2
6
2
2
2
2
6
2
2
4
4
4
2
1
4
3
2
4
4
4
1
4
4
4
4
1
4
3
4
4
3
4
3
3
2
1
4
2
1
2
4
2
4
4
4
4
4
4
4
1
4
4
4
4
4
4
4
4
2
4
2
4
4
4
2
4
4
2
4
4
3
2
1
4
4
4
2
2
2
4
4
4
4
4
4
4
4
2
4
4
2
6
2
2
4
4
4
4
4
4
4
4
1
4
4
4
4
4
4
4
2
4
4
4
4
4
4
4
2
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
2
4
4
4
2
4
1
1
1
1
4
4
4
4
4
4
4
4
4
4
4
4


In [4]:
# collect atom-level labels from train split
labels = []
for batch in train_loader:
    labels.append(batch.y.argmax(dim=1))  # remove argmax if y is already class indices

labels = torch.cat(labels)

# class counts
class_counts = torch.bincount(labels, minlength=5).float()

# sqrt inverse frequency
class_weights = 1.0 / torch.sqrt(class_counts)

# normalize so mean weight = 1
class_weights = class_weights / class_weights.mean()

# move to device
class_weights = class_weights.to(labels.device)

# model instance
model = GINE(
    input_dim=9,
    hidden_dim=128,
    output_dim=5,
    edge_dim=8,
    num_layers=3
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=5e-4,
    weight_decay=1e-4
)

criterion = torch.nn.CrossEntropyLoss(weight=class_weights)


In [5]:
counter = 0
patience = 20
best_val_loss = float('inf')
num_epochs = 1000

for epoch in range(num_epochs):
    # training
    model.train()
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        out = model(batch.x, batch.edge_index, batch.edge_attr)
        loss = criterion(out, batch.y.argmax(dim=1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_train_loss = total_loss / len(train_loader)

    # validation
    model.eval()
    val_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for batch in val_loader:
            out = model(batch.x, batch.edge_index, batch.edge_attr)
            loss = criterion(out, batch.y.argmax(dim=1))
            val_loss += loss.item()

            preds, true = split_batch_by_molecule(out, batch)
            for p, t in zip(preds, true):
                if torch.equal(p, t):
                    correct += 1
                total += 1
    avg_val_loss = val_loss / len(val_loader)

    # validation accuracy
    val_acc = correct / total
    print(f"Epoch {epoch+1} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc*100:.2f}%")

    # early stopping
    if avg_val_loss < best_val_loss:
        counter = 0
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), f"../models/gine2/gine_{epoch+1}.pt")
    else:
        counter += 1
        if counter >= patience:
            print(f"Training stopped at epoch {epoch+1}.")
            break

Epoch 1 | Train Loss: 1.4161 | Val Loss: 1.2424 | Val Acc: 13.52%
Epoch 2 | Train Loss: 1.1466 | Val Loss: 1.0269 | Val Acc: 25.07%
Epoch 3 | Train Loss: 0.9710 | Val Loss: 0.9398 | Val Acc: 23.14%
Epoch 4 | Train Loss: 0.8788 | Val Loss: 0.8886 | Val Acc: 23.68%
Epoch 5 | Train Loss: 0.8315 | Val Loss: 0.7858 | Val Acc: 26.14%
Epoch 6 | Train Loss: 0.7652 | Val Loss: 0.7343 | Val Acc: 27.37%
Epoch 7 | Train Loss: 0.7269 | Val Loss: 0.6925 | Val Acc: 28.81%
Epoch 8 | Train Loss: 0.6755 | Val Loss: 0.6999 | Val Acc: 29.29%
Epoch 9 | Train Loss: 0.6428 | Val Loss: 0.6054 | Val Acc: 32.87%
Epoch 10 | Train Loss: 0.5964 | Val Loss: 0.5902 | Val Acc: 33.78%
Epoch 11 | Train Loss: 0.5783 | Val Loss: 0.6050 | Val Acc: 33.83%
Epoch 12 | Train Loss: 0.5468 | Val Loss: 0.5807 | Val Acc: 38.48%
Epoch 13 | Train Loss: 0.5293 | Val Loss: 0.5584 | Val Acc: 33.51%
Epoch 14 | Train Loss: 0.5086 | Val Loss: 0.6027 | Val Acc: 37.73%
Epoch 15 | Train Loss: 0.5000 | Val Loss: 0.5088 | Val Acc: 40.67%
Epoc

In [5]:
MODEL_PATH = '../models/gine2/gine_114.pt'
OUTPUT_CSV = '../results/gine_test_eval.csv'

# load best model
model.load_state_dict(torch.load(MODEL_PATH))

smiles_original = []
smiles_predicted = []
smiles_matched = []
dumb_tuple_1 = []
dumb_tuple_2 = []
dumb_tuple_3 = []

# test evaluation
model.eval()
test_loss, correct, total = 0, 0, 0
with torch.no_grad():
    for batch in test_loader:
        out = model(batch.x, batch.edge_index, batch.edge_attr)
        loss = criterion(out, batch.y.argmax(dim=1))
        test_loss += loss.item()

        preds, true, dumb_tuple = split_batch_by_molecule(out, batch)

        smiles_original.extend(batch.smiles)
        smiles_predicted.extend(get_prediction_smiles(preds, batch.smiles))

        batch_smiles_true = batch.smiles
        batch_smiles_pred = get_prediction_smiles(preds, batch.smiles)

        for d, smi_p, smi_t in zip(dumb_tuple, batch_smiles_pred, batch_smiles_true):
            try:
                if mapped_molecules_equivalent(smi_p, smi_t):
                    smiles_matched.append(True)
                    correct += 1
                else:
                    smiles_matched.append(False)
            except:
                smiles_matched.append(False)
            total += 1

            dumb_tuple_1.append(d[0])
            dumb_tuple_2.append(d[1])
            dumb_tuple_3.append(d[2])

avg_test_loss = test_loss / len(test_loader)
test_acc = correct / total

print(f"Test Loss: {avg_test_loss:.4f}")
print(f"Test Accuracy: {test_acc*100:.2f}%")

# saves test predictions to .csv file
test_df = pd.DataFrame({
    'SMILES Original': smiles_original,
    'SMILES Predicted': smiles_predicted,
    'SMILES Matched': smiles_matched,
    '0 >= 1': dumb_tuple_1,
    'Score 0': dumb_tuple_2,
    'Score 1': dumb_tuple_3
})
test_df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved predictions to {OUTPUT_CSV}.")

Test Loss: 1.9691
Test Accuracy: 36.70%
Saved predictions to ../results/gine_test_eval.csv.


In [ ]:
# 10 correct: 84.48%
# 11 correct: 96.25%
# 20 correct: 91.97%
# 21 correct: 88.92%